# 01 – Metrics & Baseline

**Phase 1 exit gate:** Unit tests pass. Validation R-precision for the baseline is recorded (expect ≈0.50).

**Deliverables:** `src/metrics.py` ✓, `tests/test_metrics.py` ✓, this notebook, baseline row in experiment log.

In [ ]:
import sys, os
import numpy as np
import torch

sys.path.insert(0, "..")
sys.path.append("/kaggle/input/food-retrieval-src")

from src.config import DATA_DIR, OUT_DIR, seed_everything, load_config, ensure_dirs
SEED = 42
seed_everything(SEED)

import torchvision, timm, sklearn
print({k: v.__version__ for k, v in dict(
    torch=torch, torchvision=torchvision, timm=timm, sklearn=sklearn, numpy=np).items()})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Run metric unit tests

In [ ]:
# AGENTS.md Hard Rule 19: tests MUST pass before reporting any number
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "../tests/test_metrics.py", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Metric tests FAILED – fix before proceeding.")
print("All metric tests passed.")

## 2. Load data

In [ ]:
import pandas as pd
from src.data import load_csvs, GalleryDataset, get_transforms, image_dir, parse_val_ground_truth

dfs = load_csvs(DATA_DIR)
train_df    = dfs["train"]
val_gallery = dfs["val_gallery"]
val_queries = dfs["val_queries"]
val_gt_df   = dfs["val_gt"]
test_gallery= dfs["test_gallery"]
test_queries= dfs["test_queries"]
meta        = dfs["category_metadata"]

val_gt = parse_val_ground_truth(val_gt_df)  # {query_id: [image_id, ...]}

## 3. Text matching (surface-form)

In [ ]:
from src.text_matching import build_surface_form_dict, TextMatcher, evaluate_text_matching

surface_forms = build_surface_form_dict(meta)
matcher = TextMatcher(surface_forms, meta_df=meta)

print("Train query accuracy:")
train_acc = evaluate_text_matching(matcher, dfs["train_queries"])
print()
print("Validation query accuracy:")
val_acc = evaluate_text_matching(matcher, val_queries)

## 4. Extract ResNet-50 features (train + val + test galleries)

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.data import GalleryDataset, get_transforms, image_dir
from src.models import load_timm_model
from src.features import extract_image_features, save_features, load_features
from src.config import OUT_DIR
import pathlib

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_TAG = "resnet50_in1k"
IMG_SIZE  = 224
FEAT_DIR  = pathlib.Path(OUT_DIR) / "features" / MODEL_TAG

# ImageNet normalisation for ResNet
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

model = load_timm_model("resnet50.a1_in1k", pretrained=True, num_classes=0, device=DEVICE)
tf_val = get_transforms("val", size=IMG_SIZE, mean=MEAN, std=STD)

for split, df in [("train", train_df), ("val", val_gallery), ("test", test_gallery)]:
    cache_path = FEAT_DIR / f"{split}_image.npy"
    if cache_path.exists():
        print(f"  {split}: cached features found, skipping extraction.")
        continue
    ds = GalleryDataset(df, image_dir(DATA_DIR, split if split != "val" else "val"), transform=tf_val)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    feats, ids = extract_image_features(model, loader, device=DEVICE)
    save_features(feats, ids, FEAT_DIR, split)

## 5. Logistic regression on frozen features

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import numpy as np
from src.features import load_features

X_train, train_ids = load_features(FEAT_DIR, "train", gallery_df=train_df)
X_val,   val_ids   = load_features(FEAT_DIR, "val",   gallery_df=val_gallery)
X_test,  test_ids  = load_features(FEAT_DIR, "test",  gallery_df=test_gallery)

# Map image_id → category_id for training labels
id2cat = dict(zip(train_df["image_id"], train_df["category_id"]))
y_train = np.array([id2cat[i] for i in train_ids])

le = LabelEncoder().fit(y_train)
y_enc = le.transform(y_train)

clf = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs",
                         multi_class="multinomial", random_state=SEED)
clf.fit(X_train, y_enc)
print(f"Train acc: {clf.score(X_train, y_enc):.4f}")

## 6. Evaluate on validation set

In [ ]:
from src.metrics import evaluate, print_metrics
from src.submit import build_submission

# Score matrix: (N_val_images, K_classes) – use predict_proba for calibrated scores
S_val = clf.predict_proba(X_val)  # (N_val, K)

# Map validation queries → category zero-based indices
cat_ids_ordered = le.classes_  # class order used by the classifier
cat2idx = {int(c): i for i, c in enumerate(cat_ids_ordered)}

pred_dict = {}
for _, row in val_queries.iterrows():
    cat_id, method, _ = matcher.predict(str(row["description"]))
    if cat_id not in cat2idx:
        continue
    cat_idx = cat2idx[cat_id]
    order = np.argsort(-S_val[:, cat_idx], kind="stable")
    pred_dict[str(row["query_id"])] = [val_ids[j] for j in order[:20]]

metrics = evaluate(pred_dict, val_gt)
print("\nValidation metrics:")
print_metrics(metrics)

## 7. Generate and save baseline submission

In [ ]:
from src.submit import build_submission, validate_submission, save_submission

S_test = clf.predict_proba(X_test)

test_query_ids, test_cat_indices = [], []
for _, row in test_queries.iterrows():
    cat_id, _, _ = matcher.predict(str(row["description"]))
    test_query_ids.append(str(row["query_id"]))
    test_cat_indices.append(cat2idx.get(cat_id, 0))

sub_df = build_submission(S_test, test_ids, test_query_ids, test_cat_indices, top=20)
is_valid = validate_submission(sub_df, test_gallery)

if is_valid:
    save_submission(sub_df, pathlib.Path(OUT_DIR) / "submissions", config_name="baseline_resnet50")

# Log to experiment log
print("\nAdd to reports/experiment_log.md:")
print(f"| E001 | {pd.Timestamp.now().date()} | 1 | ResNet-50 + logreg + surface-form | C=1.0, seed=42 | 42 | {metrics['r_precision']:.4f} | {metrics['map@20']:.4f} | {metrics['p@5']:.4f} | {metrics['hit@1']:.4f} | {metrics['mrr@20']:.4f} | baseline |")